# Solenoid decoding, per body-frame dimension, across sessions

Loads and visualises the results of
`across-sessions/bhv-decode-solenoid/decode_solenoid_per_dim.py`, which decodes **four** solenoid
label encodings from **each body-frame dimension separately** (`rc`/`vt`/`ml`), across every
session in `common_utils.ALL_SESSIONS`:

| target | classes | derived from |
|---|---|---|
| `contra_ipsi` | 2 | `Params.sol_dir_to_contra_ipsi` |
| `level` | 2 | `Params.sol_dir_to_level` ("upper vs lower") |
| `angle` | 6 | `Params.sol_dir_to_angle` |
| `direction` | 12 | `values_Sol_direction` itself, unmapped ("everything") |

Each is decoded from a fixed 0.0–1.5 s post-onset window, rotated into the body frame, with
`LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')` and 5-fold `StratifiedKFold`.

**No "all features together" option here.** An early version included one; sklearn's LDA with
`shrinkage='auto'` estimates the Ledoit-Wolf shrinkage separately *per class* before pooling, so
cost scales with `n_classes x n_features**2`. That was fine for the 2-class targets (finished in
minutes at 8550 features) but hung for **over 2 hours with no result** on the 6-class `angle`
target before being killed — so it was dropped. `decode_solenoid_direction.py` (this same folder)
already covers "all features, 12-way direction" with a moving-window decoder, and is loaded and
plotted separately in `notebooks/behaviour/kinematics_001.ipynb`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent.parent))  # repo root, for tools
from tools.params import Params  # noqa: E402

TARGETS = ["contra_ipsi", "level", "angle", "direction"]
DIMS = ["rc", "vt", "ml"]
TARGET_TITLES = {
    "contra_ipsi": "contra vs. ipsi (2-way)",
    "level": "upper vs. lower (2-way)",
    "angle": "angle (6-way)",
    "direction": "direction, unmapped (12-way, “everything”)",
}

DATA_DIR = Params.ACROSS_SESSION_RESULTS_DIR / "bhv-decode-solenoid" / "data"
print(f"loading from {DATA_DIR}")

In [ ]:
npz_files = sorted(DATA_DIR.glob("perdim_*.npz"))

# results[target][dim] -> list of dicts, one per session where that (target, dim) wasn't skipped
results = {t: {d: [] for d in DIMS} for t in TARGETS}
sessions_loaded = []

for f in npz_files:
    d = np.load(f, allow_pickle=True)
    session = str(d["session"])
    sessions_loaded.append(session)
    animal = session[:4]
    for t in TARGETS:
        nc_key = f"{t}_n_classes"
        if nc_key not in d.files:
            continue
        n_classes = int(d[nc_key])
        min_n = int(d[f"{t}_min_class_n"])
        for dim in DIMS:
            key = f"{t}_{dim}_scores"
            if key not in d.files:
                continue  # skipped: smallest class had fewer trials than CV_FOLDS
            scores = d[key]
            results[t][dim].append(dict(
                session=session, animal=animal, scores=scores, mean=float(scores.mean()),
                std=float(scores.std()), chance=float(d[f"{t}_{dim}_chance"]),
                n_classes=n_classes, min_class_n=min_n,
            ))

print(f"{len(sessions_loaded)} sessions loaded: {sessions_loaded}")
for t in TARGETS:
    n_present = len(results[t]["rc"])
    skipped = len(sessions_loaded) - n_present
    print(f"  {t:<12s} present in {n_present}/{len(sessions_loaded)} sessions"
          + (f"  ({skipped} skipped: smallest class < {5} trials)" if skipped else ""))

In [ ]:
print(f"{'target':<12s}{'dim':>5s}{'n':>4s}{'median acc':>12s}{'range':>16s}"
      f"{'chance':>8s}{'above chance':>14s}")
for t in TARGETS:
    for dim in DIMS:
        rows = results[t][dim]
        if not rows:
            print(f"{t:<12s}{dim:>5s}   -- skipped in every session --")
            continue
        means = np.array([r["mean"] for r in rows])
        chance = rows[0]["chance"]
        n_above = int((means > chance).sum())
        print(f"{t:<12s}{dim:>5s}{len(rows):>4d}{np.median(means):>12.3f}"
              f"  [{means.min():.3f}, {means.max():.3f}]{chance:>8.3f}"
              f"{n_above:>10d}/{len(rows)}")

In [ ]:
fig, axs = plt.subplots(1, len(TARGETS), figsize=(4.2 * len(TARGETS), 4.2), sharey=False)
rng = np.random.default_rng(0)
animals = sorted(set(a for t in TARGETS for dim in DIMS for r in results[t][dim]
                     for a in [r["animal"]]))
animal_color = {a: plt.cm.tab10(i % 10) for i, a in enumerate(animals)}

for ax, t in zip(axs, TARGETS):
    chance = None
    for x, dim in enumerate(DIMS):
        rows = results[t][dim]
        if not rows:
            continue
        chance = rows[0]["chance"]
        means = np.array([r["mean"] for r in rows])
        ax.errorbar([x], [means.mean()], yerr=[means.std()], fmt="o", color="k",
                    zorder=2, capsize=4, markersize=7)
        jitter = rng.normal(0, 0.06, size=len(means))
        for r, jx in zip(rows, jitter):
            ax.scatter(x + jx, r["mean"], color=animal_color[r["animal"]], s=22, zorder=3,
                       alpha=0.85, edgecolors="none")
    if chance is not None:
        ax.axhline(chance, color="k", ls="--", lw=1, label="chance", zorder=1)
    ax.set_xticks(range(len(DIMS)))
    ax.set_xticklabels(DIMS)
    ax.set_title(TARGET_TITLES[t], fontsize=9)
    ax.set_xlim(-0.5, len(DIMS) - 0.5)

axs[0].set_ylabel("accuracy (5-fold CV, post 0.0-1.5s)")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, label=a, markersize=7)
          for a, c in animal_color.items()]
axs[-1].legend(handles=handles, fontsize=7, frameon=False, loc="upper left",
              bbox_to_anchor=(1.02, 1))
fig.suptitle(f"Decoding solenoid label from a single body-frame dimension "
             f"— {len(sessions_loaded)} sessions, {len(animals)} animals", fontsize=10)
fig.tight_layout()

### Reading this

Each panel is one label encoding; within a panel, `rc`/`vt`/`ml` are decoded **separately** (never
concatenated) from the same fixed post-onset window. Black points are the across-session mean ±
sd; coloured points are individual sessions, coloured by animal, so a target/dim combo that's
driven by one animal rather than replicating across animals is visible directly rather than
averaged away.

A target/dim panel with fewer coloured points than sessions loaded means it was skipped in some
sessions -- `StratifiedKFold` needs at least `CV_FOLDS` (5) trials in the smallest class, which the
12-way `direction` target is the most likely to fail (each of the 12 codes gets a small share of
an already-perturbed-trial-only session).